In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
import matplotlib.pyplot as plt
from math import ceil, floor
from datetime import datetime 
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(123)

# Model definition

In [2]:
device = (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
print(f"Training on device {device}.")



NUM_CLASSES = 10
IMAGE_HEIGHT, IMAGE_WIDTH = 48, 60


class LocalizationCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 6 * 7, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes + 5),
        )

    def forward(self, x):
        x = self.features(x)
        return self.regressor(x)



Training on device cpu.


## Localization


In [3]:
def localization_loss(y_pred, y_true):
    obj_true = y_true[:, 0].float()

    detection_loss = F.binary_cross_entropy_with_logits(
        y_pred[:, 0], obj_true, reduction='none'
    )

    bbox_loss = F.mse_loss(
        torch.sigmoid(y_pred[:, 1:5]), y_true[:, 1:5].float(), reduction='none'
    ).mean(dim=1)

    class_targets = y_true[:, 5].long().clamp(min=0, max=y_pred.shape[1] - 6)
    class_loss = F.cross_entropy(y_pred[:, 5:], class_targets, reduction='none')

    total_loss = torch.where(
        obj_true > 0.5,
        detection_loss + bbox_loss + class_loss,
        detection_loss,
    )
    return total_loss.mean()



### Load data and preprocessing

In [ ]:
data_path = "../data_2/"

# Load datasets
train_data = torch.load(f"{data_path}localization_train.pt")
val_data = torch.load(f"{data_path}localization_val.pt")
test_data = torch.load(f"{data_path}localization_test.pt")

# Check sizes
print(len(train_data))
print(len(val_data))
print(len(test_data))

59400
6600
11000


### Normalize Images

In [5]:
def _ensure_channel_first(image):
    image = image.float()
    if image.ndim == 2:
        image = image.unsqueeze(0)
    return image


class LocalizationDataset(Dataset):
    def __init__(self, data, preprocessor=None):
        self.data = data
        self.preprocessor = preprocessor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image, target = self.data[idx]
        image = _ensure_channel_first(image)
        target = target.float()
        if self.preprocessor is not None:
            image = self.preprocessor(image)
        return image, target


train_images = torch.stack([_ensure_channel_first(x) for x, _ in train_data], dim=0)
train_mean = train_images.mean()
train_std = train_images.std().clamp_min(1e-6)
print(f"Train mean: {train_mean.item():.4f}, train std: {train_std.item():.4f}")


def preprocessor(x):
    return (x - train_mean) / train_std


batch_size = 128
train_loader = DataLoader(LocalizationDataset(train_data, preprocessor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(LocalizationDataset(val_data, preprocessor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(LocalizationDataset(test_data, preprocessor), batch_size=batch_size, shuffle=False)



Train mean: 0.4171, train std: 0.2138


### Training

In [6]:
model = LocalizationCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 12
history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
best_state = None

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0

    for images, targets in train_loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        preds = model(images)
        loss = localization_loss(preds, targets)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * images.size(0)

    train_loss = running_train_loss / len(train_loader.dataset)

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            preds = model(images)
            loss = localization_loss(preds, targets)
            running_val_loss += loss.item() * images.size(0)

    val_loss = running_val_loss / len(val_loader.dataset)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"train loss: {train_loss:.4f} | val loss: {val_loss:.4f}"
    )

if best_state is not None:
    model.load_state_dict(best_state)



KeyboardInterrupt: 

### Predictions

In [ ]:
def _bbox_center_to_rect(bb, image_height=IMAGE_HEIGHT, image_width=IMAGE_WIDTH):
    x, y, w, h = bb.tolist()
    left = (x - 0.5 * w) * image_width
    top = (y - 0.5 * h) * image_height
    width = w * image_width
    height = h * image_height
    return left, top, width, height


model.eval()
num_examples = 8
indices = torch.randperm(len(test_data))[:num_examples]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

with torch.no_grad():
    for ax, idx in zip(axes, indices):
        image, target = test_data[int(idx)]
        image = _ensure_channel_first(image)

        model_input = preprocessor(image).unsqueeze(0).to(device)
        pred = model(model_input).cpu().squeeze(0)

        pred_pc = torch.sigmoid(pred[0]).item()
        pred_label = pred[5:].argmax().item()
        pred_bbox = torch.sigmoid(pred[1:5])

        true_pc = int(target[0].item())
        true_label = int(target[5].item())
        true_bbox = target[1:5]

        ax.imshow(image.squeeze(0), cmap='gray')

        if true_pc == 1:
            l, t, w, h = _bbox_center_to_rect(true_bbox)
            ax.add_patch(plt.Rectangle((l, t), w, h, fill=False, edgecolor='lime', linewidth=2))

        if pred_pc > 0.5:
            l, t, w, h = _bbox_center_to_rect(pred_bbox)
            ax.add_patch(plt.Rectangle((l, t), w, h, fill=False, edgecolor='red', linewidth=2))

        ax.set_title(
            f"T:{true_label} P:{pred_label} | pc(T/P): {true_pc}/{pred_pc:.2f}",
            fontsize=9,
        )
        ax.axis('off')

plt.tight_layout()
plt.show()



### Model selection and evaluation

In [ ]:
def intersection(bb1, bb2):
    """
    Compute intersection between 2 bb, in global frame of ref
    bb format: [x_center, y_center, width, height]
    """
    bb1 = torch.as_tensor(bb1, dtype=torch.float32)
    bb2 = torch.as_tensor(bb2, dtype=torch.float32)

    bb1_min = bb1[:2] - 0.5 * bb1[2:]
    bb1_max = bb1[:2] + 0.5 * bb1[2:]
    bb2_min = bb2[:2] - 0.5 * bb2[2:]
    bb2_max = bb2[:2] + 0.5 * bb2[2:]

    inter_min = torch.maximum(bb1_min, bb2_min)
    inter_max = torch.minimum(bb1_max, bb2_max)
    inter_wh = (inter_max - inter_min).clamp(min=0)
    return (inter_wh[0] * inter_wh[1]).item()


def IoU(bb1, bb2):
    """
    Compute IoU given 2 bb (local or global)
    """
    bb1 = torch.as_tensor(bb1, dtype=torch.float32)
    bb2 = torch.as_tensor(bb2, dtype=torch.float32)

    inter = intersection(bb1, bb2)
    area1 = (bb1[2] * bb1[3]).item()
    area2 = (bb2[2] * bb2[3]).item()
    union = area1 + area2 - inter
    if union <= 0:
        return 0.0
    return inter / union


def compute_IoU_localization(model, loader, preprocessor):
    """
    Compute IoU performance of the model on the given dataset
    """
    model.eval()
    total_iou = 0.0
    total_objects = 0

    with torch.no_grad():
        for images, targets in loader:
            if preprocessor is not None:
                images = preprocessor(images)

            images = images.to(device)
            targets = targets.to(device)
            preds = model(images)

            true_obj = targets[:, 0] > 0.5
            if not true_obj.any():
                continue

            pred_obj = torch.sigmoid(preds[:, 0]) > 0.5
            pred_bbox = torch.sigmoid(preds[:, 1:5])
            true_bbox = targets[:, 1:5]

            pred_bbox = pred_bbox[true_obj]
            true_bbox = true_bbox[true_obj]
            detected = pred_obj[true_obj]

            pred_min = pred_bbox[:, :2] - 0.5 * pred_bbox[:, 2:]
            pred_max = pred_bbox[:, :2] + 0.5 * pred_bbox[:, 2:]
            true_min = true_bbox[:, :2] - 0.5 * true_bbox[:, 2:]
            true_max = true_bbox[:, :2] + 0.5 * true_bbox[:, 2:]

            inter_min = torch.maximum(pred_min, true_min)
            inter_max = torch.minimum(pred_max, true_max)
            inter_wh = (inter_max - inter_min).clamp(min=0)
            inter_area = inter_wh[:, 0] * inter_wh[:, 1]

            pred_area = (pred_bbox[:, 2] * pred_bbox[:, 3]).clamp(min=0)
            true_area = (true_bbox[:, 2] * true_bbox[:, 3]).clamp(min=0)
            union_area = pred_area + true_area - inter_area

            iou = torch.where(union_area > 0, inter_area / union_area, torch.zeros_like(union_area))
            iou = iou * detected.float()

            total_iou += iou.sum().item()
            total_objects += true_obj.sum().item()

    if total_objects == 0:
        return 0.0
    return total_iou / total_objects


def compute_accuracy_localization(model, loader, preprocessor):
    """
    Compute accuracy of the model on the given dataset
    """
    model.eval()
    total_objects = 0
    correct = 0

    with torch.no_grad():
        for images, targets in loader:
            if preprocessor is not None:
                images = preprocessor(images)

            images = images.to(device)
            targets = targets.to(device)
            preds = model(images)

            true_obj = targets[:, 0] > 0.5
            if not true_obj.any():
                continue

            pred_obj = torch.sigmoid(preds[:, 0]) > 0.5
            pred_cls = preds[:, 5:].argmax(dim=1)
            true_cls = targets[:, 5].long()

            batch_correct = pred_obj & (pred_cls == true_cls) & true_obj
            correct += batch_correct.sum().item()
            total_objects += true_obj.sum().item()

    if total_objects == 0:
        return 0.0
    return correct / total_objects


val_acc = compute_accuracy_localization(model, val_loader, preprocessor=None)
val_iou = compute_IoU_localization(model, val_loader, preprocessor=None)
val_overall = 0.5 * (val_acc + val_iou)

print(f"Validation accuracy: {val_acc:.4f}")
print(f"Validation IoU:      {val_iou:.4f}")
print(f"Validation overall:  {val_overall:.4f}")

test_acc = compute_accuracy_localization(model, test_loader, preprocessor=None)
test_iou = compute_IoU_localization(model, test_loader, preprocessor=None)
test_overall = 0.5 * (test_acc + test_iou)

print(f"Test accuracy: {test_acc:.4f}")
print(f"Test IoU:      {test_iou:.4f}")
print(f"Test overall:  {test_overall:.4f}")

